> labeling front pin using manual and automation method

In [ ]:
#| default_exp labeling_agent.missing_single_pin

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import numpy as np
from PIL import Image
from fastcore.imports import *
from fastcore.all import *
from tqdm.auto import tqdm
import cv2
import shutil
import json
from typing import NamedTuple, Callable, Union, List

In [ ]:
ada_tools_path = Path(r'/home/ai_sintercra/homes/hasan/projects/ada_tools')
sys.path.append(str(ada_tools_path))
cv_tools_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(cv_tools_path))

In [ ]:
from  cv_tools.core import *
from  cv_tools.cv_ops import *

In [ ]:
import re

# Label studio problem chekcing

In [ ]:
def read_json(json_file_name:str):
    with open (json_file_name, 'r') as f:
        data = json.load(f)
    return data
class LabelStudioTask(NamedTuple):
    data: dict
    annotations: List[dict]

    

class LabelStudioAnnotation(NamedTuple):
    id:int
    completed_by:int
    result:str


class LabelStudioProject(NamedTuple):
    tasks: List[LabelStudioTask]
    annotations: List[dict]

def read_json(json_file_name:str):
    with open (json_file_name, 'r') as f:
        data = json.load(f)
    return data

In [ ]:
def get_name_from_data(
                    sn_task:LabelStudioTask,# annotation['data'] 
                    processing_:Union[Callable, None]=None, # which fn needs to be applied to match local and label studio name
                    ):
    "getting Name of the file from label stuido annotation"
    try:
        nm = sn_task['file_upload'].split('/')[-1].split('-')[-1]

    except:
        nm = sn_task['data']['image'].split('/')[-1]
    if processing_:
        return processing_(nm)
    else: return nm

In [ ]:
def get_hw(sn_task:LabelStudioTask):
    'Get height and width of the task'
    h = sn_task['annotations'][0]['result'][0]['original_height']
    w = sn_task['annotations'][0]['result'][0]['original_width']
    return h, w

In [ ]:
# checking whether problem with file name or label studio converter

In [ ]:
from platform import system
if system() == 'Windows':
    json_path = Path(r'm:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\missing_sn_images\third_t\json_data\project-3-at-2024-09-16-07-41-217fcf9e.json')
    im_path = Path(r'm:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\missing_sn_images\third_t')
else:
    json_path = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/third_t/json_data/project-3-at-2024-09-16-07-41-217fcf9e.json')
    im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/third_t')

In [ ]:
json_path.parent.ls()

In [ ]:
#| eval: false
#json_path = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/json/project-18-at-2024-09-12-13-30-2b2a43ab_last.json')
json_data = read_json(json_path)#

In [ ]:
fn = Path(r'In_109_idx_14.png')
im_fn = Path(im_path, fn)

In [ ]:

if system() == 'Windows':
    overlay_3 = Path(r'm:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\missing_sn_images\third_t_masks__overlay_mask')
    image3_path = Path(r'm:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\missing_sn_images\third_t')
    mask3_path = Path(image3_path.parent, 'third_t_masks_')
else:
    overlay_3 = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/third_t_masks__overlay_mask_')
    image3_path = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/third_t')
    mask3_path = Path(image3_path.parent, 'third_type_masks_')

In [ ]:
overlay_3.ls(), image3_path.ls(), mask3_path.ls()

In [ ]:
for i in mask3_path.ls():
    name_ = Path(i).name
    im_path = Path(image3_path, name_)
    
    overlay_mask(im_path, i)
    break

In [ ]:
json_data_ = []
for i in json_data:
    name_ = get_name_from_data(i, processing_=None)
    if name_ in get_name_(mask3_path.ls()):
        json_data_.append(i)

In [ ]:
get_name_from_data(json_data_[0], processing_=None)

In [ ]:
json_data_[0]

In [ ]:
process_label_studio_json(
    json_data_,
    output_dir=mask3_path
)

In [ ]:
json_data_

In [ ]:
json_data_[0]

In [ ]:
np.int32

# Note: np.int0 is not available in recent NumPy versions.
# For image coordinates, we typically use np.int32.
# This ensures compatibility across different systems and provides
# sufficient range for most image processing tasks.

In [ ]:
cv2.imwrite(f'{mask3_path}/{get_name_from_data(json_data_[0], processing_=None)}', create_mask_from_annotation(json_data_[0]['annotations'][0],106, 106 ))

In [ ]:
overlay_mask(
    Path(image3_path, get_name_from_data(json_data_[0], processing_=None)), 
    Path(mask3_path, get_name_from_data(json_data_[0], processing_=None)))

In [ ]:
get_name_from_data(json_data_[0], processing_=None)

In [ ]:
annotation = json_data_[0]['annotations'][0]

In [ ]:
import json
import numpy as np
import cv2
import os
from tqdm import tqdm

def create_mask_from_annotation(annotation, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)
    
    for result in annotation['result']:
        if result['type'] == 'rectanglelabels':
            rect = result['value']
            center_x = rect['x'] * width / 100
            center_y = rect['y'] * height / 100
            rect_width = rect['width'] * width / 100
            rect_height = rect['height'] * height / 100
            angle = rect['rotation']

            box = cv2.boxPoints(((center_x, center_y), (rect_width, rect_height), angle))
            box = np.int32(box)
            cv2.drawContours(mask, [box], 0, 255, -1)
    
    return mask

def process_label_studio_json(json_file, output_dir):
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    os.makedirs(output_dir, exist_ok=True)
    
    for task in tqdm(data, desc="Processing tasks"):
        if 'annotations' in task and task['annotations']:
            annotation = task['annotations'][0]
            width = annotation['result'][0]['original_width']
            height = annotation['result'][0]['original_height']
            
            mask = create_mask_from_annotation(annotation, width, height)
            
            # Get the image filename from the task data
            if 'file_upload' in task['data']:
                image_filename = os.path.basename(task['data']['file_upload'])
            elif 'image' in task['data']:
                image_filename = os.path.basename(task['data']['image'])
            else:
                continue  # Skip if no image filename found
            
            # Create mask filename
            mask_filename = f"{os.path.splitext(image_filename)[0]}_mask.png"
            mask_path = os.path.join(output_dir, mask_filename)
            
            # Save the mask
            cv2.imwrite(mask_path, mask)

In [ ]:

# Usage
json_file = 'path/to/your/label_studio_export.json'
output_dir = 'path/to/output/masks'

process_label_studio_json(json_file, output_dir)

In [ ]:
len(json_data_)

In [ ]:
fnames_ = []
for i in json_data:
    name_ = get_name_from_data(i, processing_=None)
    fnames_.append(name_)

In [ ]:
len(fnames_)

In [ ]:
len(list(set(fnames_)))

In [ ]:
#| eval: false
image_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/one_type')
mask_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/one_type_masks')

In [ ]:
mask_path.mkdir(exist_ok=True, parents=True)

In [ ]:
#| hide
#| eval:false
images = image_path.ls()

In [ ]:
#| hide
#| eval: false
smpl_fn = image_path.ls()[0].name

In [ ]:
#| hide
#| eval:false
pat=re.compile(r'(In|Out)_(\d+)')

In [ ]:
#| hide
#| eval: false
match_ = pat.search(smpl_fn)
match_.group(1), match_.group(2)

In [ ]:
#| hide
#| eval: false
#for idx, i in enumerate(image_path.ls()):
#    name_ = Path(i).name
#    match = pat.search(name_)
#    if match:
#        new_name = f'{match.group(1)}_{match.group(2)}_idx_{idx}{i.suffix}'
#        ne_fn = Path(image_path, new_name)
#        i.rename(ne_fn)
#    else:
#        new_name = f'idx_{idx}{i.suffix}'
#        ne_fn = Path(image_path, new_name)
#        i.rename(ne_fn)

In [ ]:
def sn_output_to_box(
                     sn_output:LabelStudioTask
                     ):
    try:
        x = (sn_output['value']['x']) * sn_output['original_width'] / 100
        y = (sn_output['value']['y']) * sn_output['original_height'] / 100
        w = sn_output['value']['width'] * sn_output['original_width'] / 100
        r = sn_output['value']['rotation'] / (180) * math.pi
        h =  sn_output['value']['height'] * sn_output['original_height'] / 100

        first_point = np.array([x,y])
        second_point = np.array([first_point[0]  + math.cos(r+0/2*math.pi) * w, first_point[1]  + math.sin(r+0/2*math.pi) * w])
        third_point  = np.array([second_point[0] + math.cos(r+1/2*math.pi) * h, second_point[1] + math.sin(r+1/2*math.pi) * h])
        fourth_point = np.array([third_point[0]  + math.cos(r+2/2*math.pi) * w, third_point[1]  + math.sin(r+2/2*math.pi) * w])
    
        return np.array([first_point, second_point, third_point, fourth_point], dtype=int)
    except Exception as e:
        print(e)

In [ ]:
def mask_gen(img, result:Dict):    
    new_img = np.zeros_like(img,dtype=np.uint8)
    for i in result:

        box = sn_output_to_box(i)
        cv2.drawContours(new_img, [box], 0,(255),-1)
    return new_img

In [ ]:
des = Path(Path(image_path).parent, 'all_single_images')
des.mkdir(exist_ok=True, parents=True)

In [ ]:
import shutil

In [ ]:
image_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/one_type')
for i in tqdm(image_path.ls(), total=len(image_path.ls())):
    parent_name = Path(i).parent.name
    new_idx = parent_name
    new_name = f'{parent_name}_{Path(i).name}'
    print(new_name)
    shutil.copyfile(i, Path(des, new_name))
    

In [ ]:
from collections import Counter

In [ ]:
single_entry_fn = [item for item, count in Counter(all_names).items() if count <2]

In [ ]:
single_entry

In [ ]:
len(all_names)

In [ ]:
len(set(all_names))

In [ ]:

all_names = [get_name_from_data(i) for i in json_data]
#[get_name_from_data(i) for i in json_data if get_name_from_data(i) in unique_names]   

In [ ]:
#| eval: false
im_f = 'third_t'
image_path = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/{im_f}')
mask_path = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/{im_f}_masks_')
mask_path.mkdir(exist_ok=True, parents=True)
for i in json_data:
    lbl_fn = get_name_from_data(i, processing_=None)
    hl, wl = get_hw(i)
    img_fns_ = list(filter(lambda x: lbl_fn in x, get_name_(image_path.ls())))
    if len(img_fns_) > 0:
        for j in img_fns_:
            if j in single_entry_fn:
                img = read_img(Path(image_path, j))
                h, w = img.shape
                if hl == h & wl == w:
                    result = i['annotations'][0]['result']
                    mask = mask_gen(img=img, result=result)
                    #show_(mask)
                    cv2.imwrite(f'{mask_path}/{Path(j).name}', mask)

In [ ]:
len(json_data)

In [ ]:
#| hide
#| eval: false
lbl_fn = get_name_from_data(json_data[1], processing_=None)

In [ ]:
fns_ = list(filter(lambda x: lbl_fn in x, get_name_(images)))

In [ ]:
for i in fns_:
    img = read_img(Path(image_path, i))
    hl, wl = get_hw(json_data[0])
    h, w = img.shape
    if hl == h & wl== w:
        print('yes')

In [ ]:
for i in images:
    name_ = Path(i).name
    

In [ ]:
json_data[0]['annotations'][0]['result'][0]

In [ ]:
#| hide
#| eval: false
h, w=get_hw(json_data[0])

In [ ]:

path = Path('/home/ai_easypid.work/data/projects/easy_front_pin_detection/eberhard_script_first')
save_path = Path('/home/ai_easypid.work/data/projects/easy_front_pin_detection/eberhard_script_first_masks')
Path(save_path).mkdir(exist_ok=True)

In [ ]:
ref_msk_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/81867539/missing/missing_unzip/main_im2_cropped_masks/threshold5/failed/missing/missing_pin_sn_masks')
ref_im_path = Path(ref_msk_path.parent, 'missing_pin_sn_images')
ref_images = get_name_(ref_im_path.ls())
im_names = set(get_name_(ref_msk_path.ls())).intersection(set(ref_images))
ref_image = Path(ref_im_path, list(im_names)[0])
ref_mask = Path(ref_msk_path, list(im_names)[0])

In [ ]:
Path(im_dst).mkdir(exist_ok=True)   

In [ ]:
im_path  = ref_im_path
im_dst = Path(im_path.parent, 'missing_pin_sn_images_dst')

In [ ]:
[shutil.copyfile(Path(im_path, i), Path(im_dst, i)) for i in tqdm(im_names)]

In [ ]:
ref_im_path.ls(), ref_msk_path.ls()

In [ ]:
ada_tools_path = Path(r'/home/ai_sintercra/homes/hasan/projects/ada_tools')
cv_tools_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')

In [ ]:
sys.path.append(str(ada_tools_path))
sys.path.append(str(cv_tools_path))

# PerSAM

# Loading SAM model


In [ ]:

from ada_tools.labeling_agent.PerSAM import *

In [ ]:

from transformers import AutoProcessor, SamModel
from huggingface_hub import hf_hub_download
model_name = "facebook/sam-vit-huge"
#model_name = "facebook/sam-vit-base"
#model_name = "facebook/sam-vit-large"
DESIRED_SIZE = 128
PROCESSOR = AutoProcessor.from_pretrained(model_name)
MODEL = SamModel.from_pretrained(
    cache_dir='/home/ai_warstein/data/huggingface/hub',
    pretrained_model_name_or_path=model_name)

In [ ]:
#ref_img = read_img(ref_im_path.ls()[0], cv=False, gray=False)
#ref_msk = read_img(ref_msk_path.ls()[0], cv=True, gray=False)

ref_img = read_img(ref_image, cv=False, gray=False)
ref_msk = read_img(ref_mask, cv=True, gray=False)

In [ ]:
ref_img

In [ ]:
show_(ref_msk)

In [ ]:
path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/81867539/missing/missing_unzip/main_im2_cropped_masks/threshold5/failed/missing/missing_pin_sn_images')
save_path = Path(Path(path).parent, 'missing_pin_sn_masks_persam')
save_path.mkdir(exist_ok=True, parents=True)


In [ ]:

# Initializing a class for PerSAM model
p_cls = perSAM_(
    model=MODEL,
    processor=PROCESSOR,
    device=None,
    ref_image=ref_img,
    ref_mask=ref_msk,
    desired_size=DESIRED_SIZE
)

# This will create mask of all images defined in folder_path and save them in save_path
# in case of best IoU is not the best mask, define best_idx
p_cls.use_persam_in_folder(
    folder_path=path,
    save_path=save_path,
    best_idx=None
)

# Checking PerSAM masks

In [ ]:
from cv_tools.dataset_check import *

In [ ]:
overlay_mask_path=Path(r'/home/ai_easypid.work/data/projects/easy_front_pin_detection/overlay_masks')
wrong_masks=Path(r'/home/ai_easypid.work/data/projects/easy_front_pin_detection/wrong_masks')
wrong_masks.mkdir(exist_ok=True, parents=True)

In [ ]:
display_image_row(
    im_path=overlay_mask_path,
    move_path=wrong_masks,
    max_images=10,
    start=0,
    im_height=200,
    im_width=200,

)

In [ ]:
from ipywidgets import (widgets, Button,
                        HBox, VBox, Layout)
from ipywidgets import Image as WImage
from IPython.display import display, clear_output
import shutil

In [ ]:
def display_image_row(
        im_path:Union[Path, str],
        move_path:Union[Path, str],
        max_images:int=10,
        start:int=0,
        im_height:int=100,
        im_width:int=100,
    ): 


    Path(move_path).mkdir(exist_ok=True, parents=True)

    def on_next_click(b):
        nonlocal start

        start = start + max_images
        print(f' start = {start}')
        clear_output(wait=True)
        display_image_row(
            im_path,
            move_path,
            max_images,
            start, 
            im_height,
            im_width
        )
    def on_prev_click(b):
        nonlocal start
        start = max( 0, start - max_images)
        clear_output(wait=True)
        display_image_row(
            im_path,
            move_path,
            max_images,
            start,
            im_height,
            im_width
        )
    def on_move_click(
            b,
            file_name):
        source_path = Path(im_path, file_name)
        destination_path = Path(move_path, file_name)
        shutil.move(source_path, destination_path)
        clear_output(wait=True)
        display_image_row(
            im_path,
            move_path,
            max_images,
            start,
            im_height,
            im_width
        )
    clear_output(wait=True)

    files = sorted(im_path.ls(file_exts=['.png', '.tif']))
    files_to_d = files[start:start + max_images]

    image_boxes = []
    
    for file in files_to_d:
        name_ = Path(file).name
        image_path = Path(im_path, name_)

        with Image.open(image_path) as pil_img:
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='PNG')
            img_widget = WImage(
                value=img_byte_arr.getvalue(),
                format='png',
                width=im_width,
                height=im_height
                )
            
            btn_del = Button(description='Delete')
            btn_move = Button(description='Move')

            btn_move.on_click(
                lambda b, f=name_: on_move_click(b, f)
            )

            box = VBox(
                [
                    img_widget,
                    HBox([btn_del, btn_move])
                    ])
            image_boxes.append(box)


    btn_next = Button(description='Next') 
    btn_prev = Button(description='Prev') 

    if start > 0 : 
        print(f'start = {start}')
        print(f'max = {max_images}')
        btn_prev.on_click(on_prev_click)
    if start + max_images < len(files):
        btn_next.on_click(on_next_click)

    nav = HBox([btn_prev, btn_next])
    display(
        HBox(image_boxes, Layout=Layout(
            flex_flow='row_wrap', 
            align_items='center',
        ))
    )
    display(nav)

# Removing images and masks which was not correct

In [ ]:
from cv_tools.core import *
from tqdm.notebook import tqdm

In [ ]:
correct_images = get_name_(overlay_mask_path.ls())

In [ ]:
trn_im_path=Path(path.parent, 'trn_imgs')
trn_msk_path = Path(path.parent, 'trn_msks')
trn_im_path.mkdir(exist_ok=True, parents=True)
trn_msk_path.mkdir(exist_ok=True, parents=True)

In [ ]:
im_path

In [ ]:
#for i in tqdm(correct_images, total=len(correct_images)):
    #im_dst = Path(trn_im_path, i)
    #im_src = Path(path, i) 
    #shutil.copyfile(im_src, im_dst)

In [ ]:
msk_path=Path("/home/ai_easypid.work/data/projects/easy_front_pin_detection/eberhard_script_first_masks_vithuge")

In [ ]:

for i in tqdm(correct_images, total=len(correct_images)):
    msk_dst = Path(trn_msk_path, i)
    msk_src = Path(msk_path, i) 
    shutil.copyfile(msk_src, msk_dst)

# Congrats !! now training is done

# Searching for missing single images which hasa masks

In [ ]:
image_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/missing_pin_sn_images')
mask_path = Path(image_path.parent, 'missing_pin_sn_masks')
mask_images = mask_path.ls()
mask_names = get_name_(mask_images)
image_names = get_name_(Path(image_path).ls())
common_mask = list(set(mask_names).intersection(set(image_names)))

In [ ]:
common_mask

In [ ]:
sm_img = Path(image_path, common_mask[0])
show_(sm_img)

In [ ]:
sin_pin_ref_image = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/missing_pin_ref_image')
sin_pin_ref_mask = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/missing_pin_ref_mask')
Path(sin_pin_ref_image).mkdir(exist_ok=True,parents=True)
Path(sin_pin_ref_mask).mkdir(exist_ok=True, parents=True)

In [ ]:
sample_im = common_mask[0]

In [ ]:
#shutil.copyfile(Path(image_path, sample_im), Path(sin_pin_ref_image, sample_im))
#shutil.copyfile(Path(mask_path, sample_im), Path(sin_pin_ref_mask, sample_im))

# Label studio file name correciton

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('29_labeling_agent.missing_single_pin.ipynb')